# Build the PCFG from the Penn Treebank

Induces a probabilistic context-free grammar from the NLTK Penn Treebank sample,
using **the same 90/10 train/test split as the HMM tagger** (`train_tagger.ipynb`, seed 42),
and checks it on the held-out treebank sentences with the standard PARSEVAL metrics.
The grammar is saved to `../models/pcfg_grammar.txt`.

In [1]:
import sys, time, json, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')
from nltk.corpus import treebank
from src.data_utils import train_test_indices
from src.pcfg_parser import (clean_tree, prepare_tree, debinarize, induce_grammar, save_grammar,
                             load_grammar, ViterbiCKYParser, parseval, brackets, tree_pos)

trees = treebank.parsed_sents()
train_idx, test_idx = train_test_indices(len(trees))
train_trees = [trees[i] for i in train_idx]
test_trees = [clean_tree(trees[i]) for i in test_idx]
print(f"train trees: {len(train_trees)}, held-out trees: {len(test_trees)}")

train trees: 3522, held-out trees: 392


## 1. Tree preprocessing

Each gold tree is cleaned before counting rules:
* `-NONE-` empty elements (traces such as `*T*-1`) are deleted, together with any constituent left empty;
* function tags and co-indices are stripped (`NP-SBJ-1` → `NP`), since they are not predicted;
* a `TOP` node is added so every tree has the same start symbol;
* the tree is binarized (Chomsky normal form with horizontal Markovization) and unary chains between
  phrases are collapsed (`S+VP`), so CKY can parse with it.

In [2]:
example = trees[train_idx[3]]
print('Original treebank tree:'); example.pretty_print()
print('After cleaning + binarization:'); prepare_tree(example, 1, 1).pretty_print()

Original treebank tree:
                                      S                                              
   ___________________________________|____________________________________________   
  |                                   VP                                           | 
  |          _________________________|__________________                          |  
  |         |             |   |                          VP                        | 
  |         |             |   |             _____________|______________           |  
  |         VP            |   |          PP-LOC               |         |          | 
  |      ___|______       |   |    ________|_______           |         |          |  
NP-SBJ  |       ADJP-PRD  |   |   |                NP         |         NP         | 
  |     |          |      |   |   |    ____________|_____     |      ___|____      |  
 PRP   VBD         JJ     CC  ,   IN  CD           JJ    NN  VBD    DT       NN    . 
  |     |          |     

## 2. Choosing the grammar settings

A plain treebank PCFG makes very strong independence assumptions. Two standard fixes are
**horizontal Markovization** (remember only the last *h* sisters when binarizing, which smooths long,
flat rules) and **vertical Markovization / parent annotation** (split each label by its parent, e.g.
`NP^S` vs `NP^VP`, which weakens the context-free assumption; Klein & Manning, 2003).
Each setting is scored on the held-out treebank sentences of up to 40 tokens.

In [3]:
dev = [t for t in test_trees if len(t.leaves()) <= 40]

def evaluate_parser(parser, gold_trees, tag_source=None):
    """PARSEVAL labeled bracket P/R/F1 (+ tagging accuracy, coverage) on gold trees.
    tag_source: None -> the parser tags words itself; else a function words -> tags."""
    matched = n_gold = n_pred = 0; failures = 0; tag_ok = n_tok = 0; start = time.time()
    for gold in gold_trees:
        words = gold.leaves()
        tree, _ = parser.parse(words, tag_source(words) if tag_source else None)
        n_gold += sum(brackets(gold).values())
        if tree is None:
            failures += 1; continue
        pred = debinarize(tree)
        m, _, p = parseval(gold, pred); matched += m; n_pred += p
        tag_ok += sum(a == b for a, b in zip(tree_pos(pred), tree_pos(gold))); n_tok += len(words)
    P, R = matched / n_pred, matched / n_gold
    return {'sentences': len(gold_trees), 'labeled_precision': P, 'labeled_recall': R,
            'labeled_f1': 2 * P * R / (P + R), 'coverage': 1 - failures / len(gold_trees),
            'tagging_accuracy': tag_ok / n_tok, 'seconds': time.time() - start}

ablation = []
for h, v in [(1, 0), (2, 0), (1, 1), (2, 1)]:
    g = induce_grammar(train_trees, horz_markov=h, vert_markov=v)
    r = evaluate_parser(ViterbiCKYParser(g, beam=40), dev)
    ablation.append({'horz_markov': h, 'vert_markov': v, 'rules': len(g.productions()), **r})
    print(f"h={h} v={v} rules={len(g.productions()):6d}  P={r['labeled_precision']:.3f} "
          f"R={r['labeled_recall']:.3f} F1={r['labeled_f1']:.3f} tag-acc={r['tagging_accuracy']:.3f} "
          f"coverage={r['coverage']:.3f}  ({r['seconds']:.0f}s)")

h=1 v=0 rules= 15990  P=0.755 R=0.722 F1=0.738 tag-acc=0.947 coverage=0.997  (163s)


h=2 v=0 rules= 17686  P=0.760 R=0.708 F1=0.733 tag-acc=0.950 coverage=0.972  (133s)


h=1 v=1 rules= 18628  P=0.785 R=0.726 F1=0.755 tag-acc=0.957 coverage=0.992  (138s)


h=2 v=1 rules= 20808  P=0.772 R=0.716 F1=0.743 tag-acc=0.955 coverage=0.989  (145s)


## 3. Final grammar

Settings used from here on: horizontal Markovization **h = 1**, parent annotation **v = 1**,
beam of **40** labels per chart cell.

In [4]:
HORZ, VERT, BEAM = 1, 1, 40
grammar = induce_grammar(train_trees, horz_markov=HORZ, vert_markov=VERT)
prods = grammar.productions()
lexical = [p for p in prods if p.is_lexical()]
binary = [p for p in prods if len(p.rhs()) == 2]
unary = [p for p in prods if not p.is_lexical() and len(p.rhs()) == 1]
print(f"rules: {len(prods)}  (binary {len(binary)}, unary {len(unary)}, lexical {len(lexical)})")
print(f"nonterminals: {len({p.lhs() for p in prods})}, POS tags: {len({p.lhs() for p in lexical})}")
print(f"unknown-word signature rules: {sum(p.rhs()[0].startswith('UNK') for p in lexical)}")
print("\nMost probable expansions of S:")
for p in sorted(grammar.productions(lhs=grammar.start().__class__('S')), key=lambda p: -p.prob())[:8]:
    print(f"  {p}")

rules: 18628  (binary 5530, unary 265, lexical 12833)
nonterminals: 1019, POS tags: 45
unknown-word signature rules: 228

Most probable expansions of S:


In [5]:
save_grammar(grammar, '../models/pcfg_grammar.txt', header=[
    f'trained on {len(train_trees)} treebank trees (random.seed(42) 90/10 split)',
    f'binarized with horzMarkov={HORZ}, vertMarkov={VERT}; unary chains collapsed with "+"',
    'format: LHS -> RHS [probability]; terminals are JSON-quoted; UNK-* terminals are unknown-word classes'])
grammar = load_grammar('../models/pcfg_grammar.txt')   # round-trip check
print(len(grammar.productions()), 'rules reloaded from ../models/pcfg_grammar.txt')

18628 rules reloaded from ../models/pcfg_grammar.txt


## 4. PARSEVAL on the held-out treebank sentences

Gold trees exist only for the treebank, so labeled bracket precision / recall / F1 (evalb conventions:
punctuation and preterminals are not scored) is measured here, with three sources of POS tags:
* **joint**: the PCFG chooses tags itself from its lexicon;
* **HMM pipeline**: tags come from the trained HMM tagger (`hmm_tagger.pkl`);
* **gold tags**: an upper bound that isolates the grammar's own errors.

In [6]:
import pickle
with open('../models/hmm_tagger.pkl', 'rb') as f:
    hmm_tagger = pickle.load(f)
hmm_tags = lambda words: [t for _, t in hmm_tagger.tag(words)]
gold_tag_of = {tuple(t.leaves()): tree_pos(t) for t in dev}

parser = ViterbiCKYParser(grammar, beam=BEAM)
heldout = {
    'joint': evaluate_parser(parser, dev),
    'hmm_pipeline': evaluate_parser(parser, dev, hmm_tags),
    'gold_tags': evaluate_parser(parser, dev, lambda w: gold_tag_of[tuple(w)]),
}
for name, r in heldout.items():
    print(f"{name:13s} P={r['labeled_precision']:.3f} R={r['labeled_recall']:.3f} "
          f"F1={r['labeled_f1']:.3f} tag-acc={r['tagging_accuracy']:.3f} coverage={r['coverage']:.3f}")

with open('../results/treebank_parseval.json', 'w') as f:
    json.dump({'settings': {'horz_markov': HORZ, 'vert_markov': VERT, 'beam': BEAM,
                            'max_sentence_length': 40, 'rules': len(grammar.productions())},
               'ablation': ablation, 'heldout': heldout}, f, indent=2)

joint         P=0.785 R=0.726 F1=0.755 tag-acc=0.957 coverage=0.992
hmm_pipeline  P=0.673 R=0.614 F1=0.642 tag-acc=0.916 coverage=0.981
gold_tags     P=0.819 R=0.751 F1=0.784 tag-acc=1.000 coverage=0.989


## 5. Example parse

In [7]:
sample = test_trees[5]
tree, logp = parser.parse(sample.leaves())
print('log P(tree) =', round(logp, 2))
debinarize(tree).pretty_print()

log P(tree) = -18.0
            TOP    
             |      
             NP    
      _______|___   
    NNP          : 
     |           |  
ACQUISITION      : 

